In [1]:
import socket
print(socket.gethostname())

awr-2-27


In [ ]:
import pandas as pd
import xarray as xr
from anemoi.datasets import open_dataset

In [18]:
date_list = ['2022-12-29-T00:00:00', '2023-01-03-T00:00:00', '2023-01-06-T06:00:00', '2023-01-07-T12:00:00', '2023-01-11-T18:00:00', '2023-03-08-T12:00:00', '2023-03-12-T18:00:00']

for date in date_list:
    start_date = pd.Timestamp(date)
    end_date = start_date + pd.Timedelta('3 days')
    print(f"Processing {start_date} to {end_date}")

    ds = open_dataset("/cw3e/mead/projects/cwp167/moerfani_data/anemoi-datasets/craft-era5-31km.zarr", start=start_date, end=end_date)

    # Pull the slice into memory — check ds.shape first to gauge size!
    data = ds[:]                      # (time, variable, ensemble, values)
    data = data[:, :, 0, :]           # drop ensemble dim if size 1

    data_vars = {
        var: (("time", "values"), data[:, i, :])
        for i, var in enumerate(ds.variables)
    }

    out = xr.Dataset(
        data_vars=data_vars,
        coords={
            "time": ds.dates,
            "latitude": ("values", ds.latitudes),
            "longitude": ("values", ds.longitudes),
        },
    )

    out.to_netcdf(f"/cw3e/mead/projects/cwp167/moerfani_data/anemoi-output/ar-analysis/TFCS-GT/{start_date.strftime('%Y-%m-%d')}.nc")

Processing 2022-12-29 00:00:00 to 2023-01-01 00:00:00
Processing 2023-01-03 00:00:00 to 2023-01-06 00:00:00
Processing 2023-01-06 06:00:00 to 2023-01-09 06:00:00
Processing 2023-01-07 12:00:00 to 2023-01-10 12:00:00
Processing 2023-01-11 18:00:00 to 2023-01-14 18:00:00
Processing 2023-03-08 12:00:00 to 2023-03-11 12:00:00
Processing 2023-03-12 18:00:00 to 2023-03-15 18:00:00
